In [1]:
import pandas as pd

ratings = pd.read_csv("https://datasets.imdbws.com/title.ratings.tsv.gz",
                      sep="\t", na_values="\\N")

colonnes = ["tconst", "titleType", "primaryTitle", "isAdult",
            "startYear", "runtimeMinutes", "genres"]
basics = pd.read_csv("https://datasets.imdbws.com/title.basics.tsv.gz",
                     sep="\t", na_values="\\N", usecols=colonnes,
                     low_memory=False)

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce")

print(basics.shape, ratings.shape)

(12958477, 7) (1731756, 3)


In [ ]:
etapes = []
films = basics.copy()
etapes.append(["Tous les titres IMDb", len(films)])

films = films[films["titleType"] == "movie"]
etapes.append(["Films de cinéma", len(films)])

films = films[films["isAdult"] == 0]
etapes.append(["Sans films adultes", len(films)])

films = films[films["startYear"] >= 1970]
etapes.append(["Depuis 1970", len(films)])

films = films[films["runtimeMinutes"] >= 60]
etapes.append(["60 minutes ou plus", len(films)])

films = films.merge(ratings, on="tconst", how="inner")
etapes.append(["Films avec une note", len(films)])

films = films[films["numVotes"] >= 10000]
etapes.append(["10 000 votes ou plus", len(films)])

films = films[films["averageRating"] >= 6]
etapes.append(["Note de 6 ou plus", len(films)])

films = films[films["genres"].str.contains("Action|Horror|Thriller|Sci-Fi|Adventure|War|Comedy", na=False)]
etapes.append(["Nos 7 genres", len(films)])

bilan = pd.DataFrame(etapes, columns=["Filtre", "Nombre de films"])
bilan

,Filtre,Nombre de films
0,Tous les titres IMDb,12958477
1,Films de cinéma,760795
2,Sans films adultes,751696
3,Depuis 1970,512778
4,60 minutes ou plus,350818
5,Films avec une note,250715
6,10 000 votes ou plus,11773
7,Note de 6 ou plus,8808
8,Nos 7 genres,6447


In [12]:
references = ["tt0076759", "tt0078748", "tt0361748", "tt0077651", "tt0073195"]
films[films["tconst"].isin(references)][["primaryTitle", "startYear", "averageRating", "numVotes", "genres"]]

,primaryTitle,startYear,averageRating,numVotes,genres
5190,Jaws,1975.0,8.1,754342,"Adventure,Drama,Horror"
7291,Star Wars: Episode IV - A New Hope,1977.0,8.6,1597875,"Action,Adventure,Fantasy"
7764,Halloween,1978.0,7.7,349583,"Horror,Thriller"
8363,Alien,1979.0,8.4,1083357,"Horror,Sci-Fi"
62774,Inglourious Basterds,2009.0,8.4,1830062,"Drama,War"


In [14]:
tmdb_ids = pd.read_csv("data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id"])

presents = films["tconst"].isin(tmdb_ids["imdb_id"]).sum()
print(presents, "films sur", len(films), "sont dans TMDB")

6255 films sur 6447 sont dans TMDB


In [15]:
films.to_csv("data/films_filtres.csv", index=False)
bilan.to_csv("data/bilan_filtres.csv", index=False)
print(len(films), "films enregistrés")

6447 films enregistrés
